# Data exploration — IMDb

Owner: Mathis. The eight questions use **PySpark** and the **silver** tables
produced by `01_data_preparation.ipynb`, through `jobs/common.py`.

Run this notebook from top to bottom after data preparation. Each result is saved
as one fixed-name CSV under `results/exploration/`. Full Parquet tables are stored
under `data/exploration/`, which is excluded from Git. Reruns overwrite these files.
Displayed tables are previews; CSV and Parquet exports contain all result rows.

In [1]:
import sys
from pathlib import Path
import csv
import json

# Locate the project from its root directory or notebooks/.
PROJECT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                if (p / "jobs" / "common.py").is_file()), None)
if PROJECT is None:
    raise FileNotFoundError("Place this notebook in the IMDb project notebooks/ directory.")
sys.path.insert(0, str(PROJECT / "jobs"))

from pyspark.sql import functions as F, Window
from pyspark.sql.types import ArrayType, MapType, StructType
from common import get_spark, read_silver, SILVER_DIR, DATASETS, DATA_DIR

spark = get_spark("imdb_data_exploration", driver_memory="2g", cores=2)
spark.sparkContext.setLogLevel("WARN")
spark.conf.set("spark.sql.shuffle.partitions", "200")

## Parameters and conventions

- Movies use `is_movie`; feature-length films use `is_feature_film` (runtime >40 minutes).
- Titles flagged `is_unreleased` are excluded. Missing ratings are never imputed.
- Q1: at least two population standard deviations from the genre mean, with at
  least 30 films per genre. Suspicious runtimes are excluded from the reference and results.
- Q2–Q3: popularity means the sum of current votes for films released in each year.
  These are not historical votes measured in that year. A production-count ranking
  is also exported to distinguish popularity from output volume.
- Q4: exactly one eligible film rated at least 8, with at least two other eligible
  films averaging at most 6; eligible ratings require at least 100 votes.
- Q5: at least three seasons with ≥10 episodes each; additional shorter seasons
  are allowed. Season 0 and unknown seasons are excluded without excluding the series.
  Episode extrema use ratings with at least 20 votes, retaining ties.
- Q6: success means a rating ≥7 and ≥1,000 votes. Original language is estimated
  from matching AKA titles and their language or region; it is not verified spoken language.
- Q7: early career means the first quarter of the observed career, with at least
  three rated films on each side and at least 100 votes per included rating.
- Q8: roles are actor/actress (combined), director, and writer.

Observed acting careers are limited to IMDb principal credits. All thresholds
are configurable below.

In [2]:
PARAMETERS = dict(
    start_year=1980, end_year=2025, runtime_z=2.0, min_genre_movies=30,
    success_rating=7.0, success_votes=1000,
    director_hit_rating=8.0, director_other_rating=6.0, director_min_votes=100,
    episode_min_votes=20,
    career_years=20, career_movies=15, early_fraction=0.25,
    career_votes=100, min_period_rated=3,
)

In [3]:
missing = [name for name in DATASETS if not (SILVER_DIR / name).is_dir()]
if missing:
    raise FileNotFoundError(
        f"Missing silver tables in {SILVER_DIR}: {missing}. "
        "Run the download job, bronze conversion and notebook 01 first.")
silver = {name: read_silver(spark, name) for name in DATASETS}
titles = silver["title.basics"].select(
    "tconst", "primaryTitle", "originalTitle", "titleType", "startYear", "runtimeMinutes", "genres",
    "is_movie", "is_feature_film", "is_unreleased", "is_runtime_suspicious")
ratings = silver["title.ratings"].select("tconst", "averageRating", "numVotes")
names = silver["name.basics"].select("nconst", "primaryName")
crew = silver["title.crew"].select("tconst", "directors", "writers")
principals = silver["title.principals"].select("tconst", "nconst", "category")
episodes = silver["title.episode"]
akas = silver["title.akas"]
movies = titles.filter(F.col("is_movie") & ~F.col("is_unreleased"))
movie_ratings = movies.join(ratings, "tconst", "left")
cast = (principals.filter(F.col("category").isin("actor", "actress"))
        .select("tconst", "nconst").dropDuplicates()
        .join(movies.select("tconst"), "tconst", "left_semi"))


RESULTS_DIR = PROJECT / "results" / "exploration"
PARQUET_DIR = DATA_DIR / "exploration"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / "parameters.json").write_text(
    json.dumps(PARAMETERS, indent=2), encoding="utf-8")
print(f"CSV results: {RESULTS_DIR}")
print(f"Full Parquet tables (not tracked by Git): {PARQUET_DIR}")

def save_result(df, name, order_by, preview=20):
    """Overwrite a fixed CSV and retain full Parquet outside version control.

    Stream rows to one CSV instead of collecting potentially large results in
    toPandas(). Array/map/struct columns are serialized as JSON for CSV export.
    """
    target = PARQUET_DIR / name
    df.write.mode("overwrite").parquet(str(target))
    stored = spark.read.parquet(str(target))
    csv_ready = stored.select(*[
        F.to_json(F.col(field.name)).alias(field.name)
        if isinstance(field.dataType, (ArrayType, MapType, StructType))
        else F.col(field.name) for field in stored.schema.fields
    ]).orderBy(*order_by)
    destination = RESULTS_DIR / f"{name}.csv"
    temporary = destination.with_suffix(".csv.tmp")
    count = 0
    with temporary.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(csv_ready.columns)
        for row in csv_ready.toLocalIterator():
            writer.writerow(tuple(row))
            count += 1
    temporary.replace(destination)
    print(f"{name}: {count:,} rows (complete CSV export)")
    stored.orderBy(*order_by).show(preview, truncate=False)
    return stored

movie_ratings.agg(
    F.count("*").alias("movies"), F.count("averageRating").alias("rated_movies"),
    F.sum(F.col("startYear").isNull().cast("long")).alias("missing_year"),
    F.sum(F.col("genres").isNull().cast("long")).alias("missing_genres"),
    F.sum(F.col("runtimeMinutes").isNull().cast("long")).alias("missing_runtime"),
).show(truncate=False)

CSV results: /home/jovyan/work/results/exploration
Full Parquet tables (not tracked by Git): /home/jovyan/work/data/exploration
+------+------------+------------+--------------+---------------+
|movies|rated_movies|missing_year|missing_genres|missing_runtime|
+------+------------+------------+--------------+---------------+
|758804|351584      |112636      |77455         |269010         |
+------+------------+------------+--------------+---------------+



## 1. Find all feature-length movies by genre whose length is notably different than the average length of movies in that genre.

One row per film–genre pair. The reference includes feature-length films of
that genre across all years, excluding `is_runtime_suspicious` films (>600 min).
`z_runtime` gives the direction and size of the difference. Genres with zero
variance or insufficient observations are excluded. This is a descriptive rule,
not a statistical normality test.

In [4]:
def unusual_runtimes(movies, z_threshold=2.0, min_count=30):
    feature_genres = (movies.filter(F.col("is_feature_film") & ~F.col("is_runtime_suspicious"))
        .withColumn("genre", F.explode(F.array_distinct("genres")))
        .filter(F.col("genre").isNotNull() & (F.trim("genre") != "")))
    stats = feature_genres.groupBy("genre").agg(
        F.count("*").alias("genre_movies"),
        F.avg("runtimeMinutes").alias("genre_mean_minutes"),
        F.stddev_pop("runtimeMinutes").alias("genre_std_minutes"))
    return (feature_genres.join(stats, "genre")
        .filter((F.col("genre_movies") >= min_count) & (F.col("genre_std_minutes") > 0))
        .withColumn("difference_minutes", F.col("runtimeMinutes") - F.col("genre_mean_minutes"))
        .withColumn("z_runtime", F.col("difference_minutes") / F.col("genre_std_minutes"))
        .filter(F.abs("z_runtime") >= z_threshold)
        .select("genre", "tconst", "primaryTitle", "startYear", "runtimeMinutes",
                "genre_movies", "genre_mean_minutes", "genre_std_minutes",
                "difference_minutes", "z_runtime", "is_runtime_suspicious"))

In [5]:
q1 = save_result(unusual_runtimes(movies, PARAMETERS["runtime_z"], PARAMETERS["min_genre_movies"]),
                 "q1_unusual_runtimes", [F.desc(F.abs("z_runtime")), "genre", "tconst"])

q1_unusual_runtimes: 30,232 rows (complete CSV export)
+-----------+----------+--------------------------------------------------+---------+--------------+------------+------------------+------------------+------------------+------------------+---------------------+
|genre      |tconst    |primaryTitle                                      |startYear|runtimeMinutes|genre_movies|genre_mean_minutes|genre_std_minutes |difference_minutes|z_runtime         |is_runtime_suspicious|
+-----------+----------+--------------------------------------------------+---------+--------------+------------+------------------+------------------+------------------+------------------+---------------------+
|Horror     |tt40792136|Growth                                            |2026     |587           |30697       |88.50118904127439 |20.386511984309774|498.4988109587256 |24.452383582948816|false                |
|Adult      |tt39394409|MILF & Cougar 3-Pack                              |2011     |560         

## 2. What are the top three most popular movie genres for each year between 1980 and 2025?

A film contributes once to each of its genres. Unrated films count toward
production but contribute zero observed votes. Alphabetical ordering breaks ties
to return exactly three genres per year where available. No vote threshold is
applied to these rankings.

In [6]:
def genre_rankings(movie_ratings, start_year=1980, end_year=2025):
    film_genres = (movie_ratings.filter(F.col("startYear").between(start_year, end_year))
        .withColumn("genre", F.explode(F.array_distinct("genres")))
        .filter(F.col("genre").isNotNull() & (F.trim("genre") != "")))
    stats = film_genres.groupBy("startYear", "genre").agg(
        F.count("*").alias("movie_count"),
        F.count("averageRating").alias("rated_movie_count"),
        F.sum(F.coalesce(F.col("numVotes"), F.lit(0)).cast("long")).alias("total_votes"),
        F.avg("averageRating").alias("mean_rating"),
        F.sum(F.col("averageRating") * F.col("numVotes")).alias("rating_vote_sum"))
    return (stats
        .withColumn("vote_weighted_rating", F.when(F.col("total_votes") > 0,
                    F.col("rating_vote_sum") / F.col("total_votes")))
        .drop("rating_vote_sum")
        .withColumn("popularity_rank", F.row_number().over(
            Window.partitionBy("startYear").orderBy(F.desc("total_votes"), "genre")))
        .withColumn("production_rank", F.row_number().over(
            Window.partitionBy("startYear").orderBy(F.desc("movie_count"), "genre"))))

In [7]:
genre_stats = save_result(genre_rankings(movie_ratings, PARAMETERS["start_year"], PARAMETERS["end_year"]),
                          "q2_all_genres", ["startYear", "popularity_rank"])
q2 = save_result(genre_stats.filter(F.col("popularity_rank") <= 3),
                 "q2_top_three_by_votes", ["startYear", "popularity_rank"], preview=138)
save_result(genre_stats.filter(F.col("production_rank") <= 3),
            "q2_top_three_by_movie_count", ["startYear", "production_rank"])
years_present = genre_stats.select("startYear").distinct()
spark.range(PARAMETERS["start_year"], PARAMETERS["end_year"] + 1).select(
    F.col("id").alias("startYear")).join(years_present, "startYear", "left_anti").show()

q2_all_genres: 1,101 rows (complete CSV export)
+---------+---------+-----------+-----------------+-----------+------------------+--------------------+---------------+---------------+
|startYear|genre    |movie_count|rated_movie_count|total_votes|mean_rating       |vote_weighted_rating|popularity_rank|production_rank|
+---------+---------+-----------+-----------------+-----------+------------------+--------------------+---------------+---------------+
|1980     |Drama    |1603       |1169             |3168139    |6.126860564585115 |7.734155887730941   |1              |1              |
|1980     |Adventure|186        |145              |2482970    |5.922758620689655 |7.9334757568556995  |2              |8              |
|1980     |Horror   |167        |151              |2074309    |5.2841059602649025|7.466729161373741   |3              |9              |
|1980     |Sci-Fi   |70         |63               |1925995    |5.23015873015873  |8.203390870692811   |4              |18             |


## 3. Compare the results from the previous question. How do the top three genres compare to the rest of the genres of movies in a given year?

Two complementary comparisons:

1. **Genres:** the top three's share of genre-level vote totals, average genre
   ratings, and votes per film–genre membership. These shares are not shares of unique films.
2. **Unique films:** a film is in the top-three group if it has at least one of
   those genres; otherwise it belongs to the remaining genres. Films without a genre
   form a separate group. These groups are disjoint, avoiding duplicate films or votes.

In [8]:
def compare_genres(genre_stats, movie_ratings, start_year=1980, end_year=2025):
    by_genre = (genre_stats.withColumn("group", F.when(F.col("popularity_rank") <= 3, "top3")
                                    .otherwise("other_genres"))
        .groupBy("startYear", "group").agg(
            F.count("*").alias("genre_count"),
            F.sum("movie_count").alias("film_genre_memberships"),
            F.sum("total_votes").alias("genre_vote_sum"),
            F.avg("mean_rating").alias("unweighted_mean_of_genre_ratings"))
        .withColumn("membership_vote_share_pct", 100 * F.col("genre_vote_sum") /
            F.when(F.sum("genre_vote_sum").over(Window.partitionBy("startYear")) > 0,
                   F.sum("genre_vote_sum").over(Window.partitionBy("startYear"))))
        .withColumn("votes_per_membership", F.col("genre_vote_sum") / F.col("film_genre_memberships")))
    top = (genre_stats.filter(F.col("popularity_rank") <= 3).groupBy("startYear")
           .agg(F.collect_set("genre").alias("top_genres")))
    grouped = (movie_ratings.filter(F.col("startYear").between(start_year, end_year))
        .join(top, "startYear", "left")
        .withColumn("group", F.when(F.col("genres").isNull() | (F.size("genres") == 0), "unknown_genre")
            .when(F.size(F.array_intersect("genres", "top_genres")) > 0, "has_top3_genre")
            .otherwise("other_genres_only")))
    by_movie = grouped.groupBy("startYear", "group").agg(
        F.count("*").alias("movie_count"), F.count("averageRating").alias("rated_movie_count"),
        F.sum(F.coalesce(F.col("numVotes"), F.lit(0)).cast("long")).alias("total_votes"),
        F.avg("averageRating").alias("mean_rating"))
    return by_genre, (by_movie
        .withColumn("movie_share_pct", 100 * F.col("movie_count") /
                    F.sum("movie_count").over(Window.partitionBy("startYear")))
        .withColumn("vote_share_pct", F.when(
            F.sum("total_votes").over(Window.partitionBy("startYear")) > 0,
            100 * F.col("total_votes") / F.sum("total_votes").over(Window.partitionBy("startYear"))))
        .withColumn("votes_per_movie", F.col("total_votes") / F.col("movie_count")))

In [9]:
q3_genres, q3_movies = compare_genres(genre_stats, movie_ratings,
                                    PARAMETERS["start_year"], PARAMETERS["end_year"])
save_result(q3_genres, "q3_genre_comparison", ["startYear", "group"])
save_result(q3_movies, "q3_disjoint_movie_comparison", ["startYear", "group"])

q3_genre_comparison: 92 rows (complete CSV export)
+---------+------------+-----------+----------------------+--------------+--------------------------------+-------------------------+--------------------+
|startYear|group       |genre_count|film_genre_memberships|genre_vote_sum|unweighted_mean_of_genre_ratings|membership_vote_share_pct|votes_per_membership|
+---------+------------+-----------+----------------------+--------------+--------------------------------+-------------------------+--------------------+
|1980     |other_genres|21         |3432                  |9728841       |6.162825435151568               |55.73906632186448        |2834.7438811188813  |
|1980     |top3        |3          |1956                  |7725418       |5.777908381846558               |44.26093367813552        |3949.6002044989773  |
|1981     |other_genres|20         |3449                  |6039631       |6.181246005605908               |53.52999600095863        |1751.1252536967236  |
|1981     |top3    

DataFrame[startYear: int, group: string, movie_count: bigint, rated_movie_count: bigint, total_votes: bigint, mean_rating: double, movie_share_pct: double, vote_share_pct: double, votes_per_movie: double]

## 4. List all directors who have directed exactly one movie with a rating >= 8.0 but who have multiple other movies directed with an average rating of <= 6.0.

Directors come from `title.crew`, not only principal credits. The single-hit
condition and the mean of the other films use ratings with at least
`director_min_votes` votes (default 100). This reduces the influence of a handful
of votes. All observed films, rated films, eligible rated films, and unrated films
are counted separately. Missing and low-vote ratings do not establish additional hits.
The other-film average is an unweighted arithmetic mean.

In [10]:
def exceptional_directors(crew, movie_ratings, names, hit_rating=8.0, other_rating=6.0, min_votes=100):
    directed = (crew.select("tconst", F.explode("directors").alias("nconst"))
                .dropDuplicates().join(movie_ratings, "tconst"))
    reliable = F.col("numVotes") >= min_votes
    high = reliable & (F.col("averageRating") >= hit_rating)
    other = reliable & (F.col("averageRating") < hit_rating)
    stats = directed.groupBy("nconst").agg(
        F.count("*").alias("all_movies"), F.count("averageRating").alias("rated_movies"),
        F.count(F.when(reliable, 1)).alias("eligible_rated_movies"),
        F.sum(F.when(high, 1).otherwise(0)).alias("high_rated_movies"),
        F.sum(F.when(other, 1).otherwise(0)).alias("other_rated_movies"),
        F.avg(F.when(other, F.col("averageRating"))).alias("other_mean_rating"))
    qualified = (stats.filter((F.col("high_rated_movies") == 1) &
                 (F.col("other_rated_movies") >= 2) & (F.col("other_mean_rating") <= other_rating))
                 .withColumn("unrated_movies", F.col("all_movies") - F.col("rated_movies")))
    hit = directed.filter(high).select("nconst", F.col("tconst").alias("hit_tconst"),
        F.col("primaryTitle").alias("hit_title"), F.col("averageRating").alias("hit_rating"),
        F.col("numVotes").alias("hit_votes"))
    return qualified.join(names, "nconst", "left").join(hit, "nconst")

In [11]:
q4 = save_result(exceptional_directors(crew, movie_ratings, names, PARAMETERS["director_hit_rating"],
    PARAMETERS["director_other_rating"], PARAMETERS["director_min_votes"]),
                 "q4_directors", ["other_mean_rating", "nconst"])

q4_directors: 183 rows (complete CSV export)
+----------+----------+------------+---------------------+-----------------+------------------+------------------+--------------+------------------+----------+------------------------------+----------+---------+
|nconst    |all_movies|rated_movies|eligible_rated_movies|high_rated_movies|other_rated_movies|other_mean_rating |unrated_movies|primaryName       |hit_tconst|hit_title                     |hit_rating|hit_votes|
+----------+----------+------------+---------------------+-----------------+------------------+------------------+--------------+------------------+----------+------------------------------+----------+---------+
|nm7263882 |13        |8           |4                    |1                |3                 |2.8000000000000003|5             |Swaylee Loughnane |tt10123970|Mother's Child                |8.2       |6734     |
|nm2840629 |14        |10          |3                    |1                |2                 |3.0         

## 5. Identify television series with at least three seasons consisting of >=10 episodes where the worst episode occurred in the last season and the best episode occurred within the first two seasons.

Season sizes and the last observed season are determined before filtering
ratings. At least three seasons must each contain ten episodes, but additional
shorter seasons are allowed, including a short final season. Season 0 and episodes
without a positive season number are ignored; their presence does not exclude a series.

The last season is the latest numbered season present in the snapshot and may still
be ongoing. The best and worst episodes are measured among ratings with at least
`episode_min_votes` votes (default 20); coverage reports the proportion of numbered
episodes meeting that requirement. Ties are retained: at least one maximum must
occur in seasons 1–2 and at least one minimum in the last season. All qualifying
extreme episodes are exported as evidence.

In [12]:
def qualifying_series(titles, episodes, ratings, min_votes=20):
    series = titles.filter(F.col("titleType").isin("tvSeries", "tvMiniSeries") &
                           ~F.col("is_unreleased")).select(
        F.col("tconst").alias("parentTconst"), F.col("primaryTitle").alias("series_title"))
    raw = episodes.join(series, "parentTconst").dropDuplicates(["tconst"])
    regular = raw.filter(F.col("seasonNumber") > 0)
    seasons = regular.groupBy("parentTconst", "seasonNumber").agg(F.count("*").alias("episodes_in_season"))
    eligible = (seasons.groupBy("parentTconst").agg(
        F.count("*").alias("season_count"),
        F.sum((F.col("episodes_in_season") >= 10).cast("int")).alias("full_seasons"),
        F.max("seasonNumber").alias("last_season"), F.sum("episodes_in_season").alias("episode_count"))
        .filter(F.col("full_seasons") >= 3))
    rated = regular.join(ratings.filter(F.col("numVotes") >= min_votes), "tconst").join(eligible, "parentTconst")
    extrema = rated.groupBy("parentTconst").agg(
        F.min("averageRating").alias("worst_rating"), F.max("averageRating").alias("best_rating"),
        F.count("*").alias("rated_episode_count"))
    marked = rated.join(extrema, "parentTconst")
    best = marked.filter((F.col("averageRating") == F.col("best_rating")) &
                          F.col("seasonNumber").isin(1, 2)).select(
        "parentTconst", "tconst", "seasonNumber", "episodeNumber", "averageRating", "numVotes")
    worst = marked.filter((F.col("averageRating") == F.col("worst_rating")) &
                           (F.col("seasonNumber") == F.col("last_season"))).select(best.columns)
    selected = best.select("parentTconst").distinct().join(worst.select("parentTconst").distinct(), "parentTconst")
    result = (eligible.join(selected, "parentTconst").join(series, "parentTconst")
        .join(extrema, "parentTconst")
        .withColumn("rated_episode_pct", 100 * F.col("rated_episode_count") / F.col("episode_count")))
    evidence = (best.withColumn("extreme", F.lit("best_in_first_two"))
        .unionByName(worst.withColumn("extreme", F.lit("worst_in_last")))
        .join(selected, "parentTconst"))
    return result, evidence

In [13]:
q5, q5_evidence = qualifying_series(titles, episodes, ratings, PARAMETERS["episode_min_votes"])
save_result(q5, "q5_series", ["parentTconst"])
save_result(q5_evidence, "q5_episode_evidence", ["parentTconst", "extreme", "tconst"])

q5_series: 412 rows (complete CSV export)
+------------+------------+------------+-----------+-------------+----------------------------------+------------+-----------+-------------------+------------------+
|parentTconst|season_count|full_seasons|last_season|episode_count|series_title                      |worst_rating|best_rating|rated_episode_count|rated_episode_pct |
+------------+------------+------------+-----------+-------------+----------------------------------+------------+-----------+-------------------+------------------+
|tt0042110   |5           |5           |5          |91           |The Gene Autry Show               |6.6         |7.9        |31                 |34.065934065934066|
|tt0043194   |8           |8           |8          |276          |Dragnet                           |6.5         |9.0        |106                |38.405797101449274|
|tt0043223   |3           |3           |3          |78           |The Range Rider                   |6.3         |7.4        |22

DataFrame[parentTconst: string, tconst: string, seasonNumber: int, episodeNumber: int, averageRating: double, numVotes: int, extreme: string]

## 6. Find any actors/actresses who have achieved success across films in multiple distinct languages (e.g. been in a successful film originally in French and English). Additionally, list all actors/actresses who have achieved success in movies in >=3 languages (e.g. French, German, and English).

**Source limitation:** IMDb's [`title.akas.language`](https://developer.imdb.com/non-commercial-datasets/)
describes a title's language, not necessarily the language spoken in the film.
The `isOriginalTitle = 1` rows do not provide usable language coverage in the group's
snapshot, so this query estimates language from non-original AKA rows whose title
matches `originalTitle` after trimming and case normalization.

Each matching AKA contributes its explicit language where usable, otherwise the
region mapping below. Regions absent from the mapping, including IN, CA, BE, CH
and ZA, contribute no regional fallback. This mapping is a heuristic, not a claim
that mapped countries are monolingual. The most frequent candidate language is
selected; alphabetical order breaks ties. The evidence includes supporting counts,
the share of matching entries and the number of tied leaders.

Outputs include candidates successful in ≥2 and ≥3 estimated languages, film-level
evidence and estimates, and coverage among successful films. Distribution practices,
unchanged titles abroad and the region fallback can bias these estimates. They need
validation against a source of original spoken languages before being treated as
confirmed multilingual careers.

In [14]:
REGION_LANG = {
    "US": "en", "GB": "en", "AU": "en", "IE": "en", "NZ": "en",
    "FR": "fr", "DE": "de", "AT": "de", "IT": "it", "ES": "es",
    "MX": "es", "AR": "es", "JP": "ja", "KR": "ko", "CN": "zh",
    "TW": "zh", "BR": "pt", "PT": "pt", "RU": "ru", "SE": "sv",
    "DK": "da", "NO": "no", "FI": "fi", "NL": "nl", "PL": "pl",
    "TR": "tr", "IR": "fa",
}

def estimate_languages(movie_ratings, akas):
    region_map = F.create_map(*[F.lit(x) for pair in REGION_LANG.items() for x in pair])
    original_titles = movie_ratings.select("tconst", "originalTitle")
    language = F.lower(F.trim("language"))
    candidates = (akas.filter(F.col("isOriginalTitle") == 0)
        .select(F.col("titleId").alias("tconst"), "title", "region", "language")
        .join(original_titles, "tconst")
        .filter(F.lower(F.trim("title")) == F.lower(F.trim("originalTitle")))
        .withColumn("explicit_language", F.when(
            language.isNotNull() & ~language.isin("", "und", "mul", "zxx", "\\n"), language))
        .withColumn("language", F.coalesce("explicit_language", region_map[F.upper(F.trim("region"))]))
        .filter(F.col("language").isNotNull()))
    counts = candidates.groupBy("tconst", "language").agg(
        F.count("*").alias("supporting_akas"),
        F.count("explicit_language").alias("explicit_language_akas"))
    window = Window.partitionBy("tconst")
    ranked = (counts
        .withColumn("total_candidate_akas", F.sum("supporting_akas").over(window))
        .withColumn("max_support", F.max("supporting_akas").over(window))
        .withColumn("tied_top_languages", F.sum(
            (F.col("supporting_akas") == F.col("max_support")).cast("int")).over(window))
        .withColumn("rank", F.row_number().over(window.orderBy(F.desc("supporting_akas"), "language"))))
    return (ranked.filter(F.col("rank") == 1).drop("rank", "max_support")
        .withColumn("support_share_pct", 100 * F.col("supporting_akas") / F.col("total_candidate_akas")))

def multilingual_candidates(cast, movie_ratings, akas, names, min_rating=7.0, min_votes=1000):
    successful = movie_ratings.filter((F.col("averageRating") >= min_rating) & (F.col("numVotes") >= min_votes))
    languages = estimate_languages(successful, akas)
    evidence = (cast.join(successful.select("tconst", "primaryTitle", "averageRating", "numVotes"), "tconst")
        .join(languages, "tconst").dropDuplicates(["nconst", "tconst", "language"]))
    summary = (evidence.groupBy("nconst").agg(
        F.countDistinct("language").alias("language_count"),
        F.sort_array(F.collect_set("language")).alias("estimated_original_languages"),
        F.countDistinct("tconst").alias("successful_movie_count"))
        .filter(F.col("language_count") >= 2).join(names, "nconst", "left")
        .withColumn("evidence_status", F.lit("estimated from matching AKA titles and regions; spoken language unverified")))
    coverage = (successful.select("tconst").join(languages.select("tconst").withColumn("has_language", F.lit(1)), "tconst", "left")
        .agg(F.count("*").alias("successful_movies"), F.count("has_language").alias("with_estimated_language"))
        .withColumn("coverage_pct", F.when(F.col("successful_movies") > 0,
            100 * F.col("with_estimated_language") / F.col("successful_movies"))))
    return summary, evidence.join(summary.select("nconst"), "nconst"), coverage, languages

In [15]:
q6, q6_evidence, q6_coverage, q6_languages = multilingual_candidates(cast, movie_ratings, akas, names,
    PARAMETERS["success_rating"], PARAMETERS["success_votes"])
save_result(q6_coverage, "q6_language_coverage", ["successful_movies"])
save_result(q6, "q6_candidates_two_or_more_languages", [F.desc("language_count"), "nconst"])
save_result(q6.filter(F.col("language_count") >= 3), "q6_candidates_three_or_more_languages", ["nconst"])
save_result(q6_evidence, "q6_language_evidence", ["nconst", "language", "tconst"])
save_result(q6_languages, "q6_movie_language_estimates", ["tconst"])

q6_language_coverage: 1 rows (complete CSV export)
+-----------------+-----------------------+-----------------+
|successful_movies|with_estimated_language|coverage_pct     |
+-----------------+-----------------------+-----------------+
|14185            |12412                  |87.50088121254846|
+-----------------+-----------------------+-----------------+

q6_candidates_two_or_more_languages: 4,623 rows (complete CSV export)
+---------+--------------+-----------------------------+----------------------+--------------------+--------------------------------------------------------------------------+
|nconst   |language_count|estimated_original_languages |successful_movie_count|primaryName         |evidence_status                                                           |
+---------+--------------+-----------------------------+----------------------+--------------------+--------------------------------------------------------------------------+
|nm0001041|7             |[de, en, es, i

DataFrame[tconst: string, language: string, supporting_akas: bigint, explicit_language_akas: bigint, total_candidate_akas: bigint, tied_top_languages: bigint, support_share_pct: double]

## 7. Find all actors/actresses who have been in movies for 20+ years and at least 15 or more movies whose popularity peaked early in their career (e.g. had their highest rated movies early in their career as compared to later.)

Career length is the last release year minus the first, across all dated films
with an actor/actress credit, including unrated films. At least 15 distinct dated
films are required. Undated films are counted separately.

An early peak requires both a strictly higher maximum rating and a higher mean
rating in the first quarter of the observed career than later. A maximum also
reached later does not qualify. Popularity is approximated by ratings, following
the question's example, rather than vote volume.

In [16]:
def early_peak_actors(cast, movie_ratings, names, min_years=20, min_movies=15,
                     early_fraction=0.25, min_votes=100, min_period_rated=3):
    filmography = cast.join(movie_ratings, "tconst")
    careers = filmography.groupBy("nconst").agg(
        F.min("startYear").alias("first_year"), F.max("startYear").alias("last_year"),
        F.count("startYear").alias("dated_movies"),
        F.sum(F.col("startYear").isNull().cast("long")).alias("undated_movies"))
    careers = (careers.withColumn("career_years", F.col("last_year") - F.col("first_year"))
        .filter((F.col("career_years") >= min_years) & (F.col("dated_movies") >= min_movies))
        .withColumn("early_end_year", F.col("first_year") + F.floor(F.col("career_years") * early_fraction)))
    observed = (filmography.join(careers, "nconst")
        .filter(F.col("startYear").isNotNull() & F.col("averageRating").isNotNull() & (F.col("numVotes") >= min_votes))
        .withColumn("early", F.col("startYear") <= F.col("early_end_year")))
    stats = observed.groupBy("nconst").agg(*[
        expr for label, condition in [("early", F.col("early")), ("later", ~F.col("early"))]
        for expr in [F.count(F.when(condition, 1)).alias(label + "_rated_movies"),
                     F.avg(F.when(condition, F.col("averageRating"))).alias(label + "_mean_rating"),
                     F.max(F.when(condition, F.col("averageRating"))).alias(label + "_max_rating")]])
    selected = (stats.filter((F.col("early_rated_movies") >= min_period_rated) &
        (F.col("later_rated_movies") >= min_period_rated) &
        (F.col("early_max_rating") > F.col("later_max_rating")) &
        (F.col("early_mean_rating") > F.col("later_mean_rating")))
        .withColumn("mean_rating_drop", F.col("early_mean_rating") - F.col("later_mean_rating")))
    peaks = (observed.join(selected.select("nconst", "early_max_rating"), "nconst")
        .filter(F.col("early") & (F.col("averageRating") == F.col("early_max_rating")))
        .select("nconst", "tconst", "primaryTitle", "startYear", "averageRating", "numVotes"))
    return selected.join(careers, "nconst").join(names, "nconst", "left"), peaks

In [17]:
q7, q7_peaks = early_peak_actors(cast, movie_ratings, names,
    PARAMETERS["career_years"], PARAMETERS["career_movies"], PARAMETERS["early_fraction"],
    PARAMETERS["career_votes"], PARAMETERS["min_period_rated"])
save_result(q7, "q7_early_peak_actors", [F.desc("mean_rating_drop"), "nconst"])
save_result(q7_peaks, "q7_peak_movies", ["nconst", "startYear", "tconst"])

q7_early_peak_actors: 4,638 rows (complete CSV export)
+---------+------------------+-----------------+----------------+------------------+------------------+----------------+------------------+----------+---------+------------+--------------+------------+--------------+--------------------+
|nconst   |early_rated_movies|early_mean_rating|early_max_rating|later_rated_movies|later_mean_rating |later_max_rating|mean_rating_drop  |first_year|last_year|dated_movies|undated_movies|career_years|early_end_year|primaryName         |
+---------+------------------+-----------------+----------------+------------------+------------------+----------------+------------------+----------+---------+------------+--------------+------------+--------------+--------------------+
|nm2767828|5                 |8.52             |8.8             |5                 |4.34              |6.3             |4.18              |1990      |2026     |35          |0             |36          |1999          |Saroj Khanal   

DataFrame[nconst: string, tconst: string, primaryTitle: string, startYear: int, averageRating: double, numVotes: int]

## 8. Identify film professionals who can be considered "multi-talented" (e.g. some combo of writer/director/actor). How do these individuals perform when performing multiple roles on the same project (e.g. directing and acting in the same movie)?

Use actual film credits rather than `primaryProfession`. Director and writer
credits from `title.crew` supplement `title.principals`. Deduplicate by person,
film and role. A professional is multi-talented if at least two distinct roles
appear across their observed filmography.

For each person, compare films with one role against films with multiple simultaneous
roles: film counts, rating coverage, mean rating, vote-weighted rating and average
votes. Individual differences require rated films in both groups. The overall
summary gives equal weight to each comparable person and is descriptive, not causal.

In [18]:
def multitalented_professionals(principals, crew, movie_ratings, names):
    principal_roles = (principals.filter(F.col("category").isin("actor", "actress", "director", "writer"))
        .select("tconst", "nconst", F.when(F.col("category").isin("actor", "actress"), "actor")
                .otherwise(F.col("category")).alias("role")))
    directors = crew.select("tconst", F.explode("directors").alias("nconst")).withColumn("role", F.lit("director"))
    writers = crew.select("tconst", F.explode("writers").alias("nconst")).withColumn("role", F.lit("writer"))
    roles = (principal_roles.unionByName(directors).unionByName(writers).dropDuplicates()
             .join(movie_ratings.select("tconst"), "tconst", "left_semi"))
    people = (roles.groupBy("nconst").agg(F.sort_array(F.collect_set("role")).alias("career_roles"))
        .filter(F.size("career_roles") >= 2).join(names, "nconst", "left"))
    projects = (roles.groupBy("nconst", "tconst").agg(F.sort_array(F.collect_set("role")).alias("project_roles"))
        .join(people.select("nconst"), "nconst")
        .withColumn("role_count", F.size("project_roles"))
        .withColumn("role_mode", F.when(F.col("role_count") >= 2, "multiple").otherwise("single"))
        .join(movie_ratings.select("tconst", "primaryTitle", "averageRating", "numVotes"), "tconst"))
    comparison = projects.groupBy("nconst", "role_mode").agg(
        F.count("*").alias("movies"), F.count("averageRating").alias("rated_movies"),
        F.avg("averageRating").alias("mean_rating"), F.avg("numVotes").alias("mean_votes_rated_movies"),
        F.sum("numVotes").alias("total_votes"),
        F.sum(F.col("averageRating") * F.col("numVotes")).alias("weighted_sum"))
    comparison = (comparison.withColumn("vote_weighted_rating", F.when(F.col("total_votes") > 0,
        F.col("weighted_sum") / F.col("total_votes"))).drop("weighted_sum")
        .withColumn("rated_pct", 100 * F.col("rated_movies") / F.col("movies"))
        .join(people, "nconst"))
    paired = comparison.groupBy("nconst").agg(
        F.max(F.when(F.col("role_mode") == "single", F.col("mean_rating"))).alias("single_mean_rating"),
        F.max(F.when(F.col("role_mode") == "multiple", F.col("mean_rating"))).alias("multiple_mean_rating"))
    paired = (paired.filter(F.col("single_mean_rating").isNotNull() & F.col("multiple_mean_rating").isNotNull())
        .withColumn("multiple_minus_single", F.col("multiple_mean_rating") - F.col("single_mean_rating"))
        .join(people, "nconst"))
    overall = paired.agg(F.count("*").alias("comparable_people"),
        F.avg("multiple_minus_single").alias("mean_person_level_rating_difference"),
        F.expr("percentile_approx(multiple_minus_single, 0.5)").alias("median_person_level_rating_difference"))
    return people, projects, comparison, paired, overall

In [19]:
q8_people, q8_projects, q8_comparison, q8_paired, q8_overall = multitalented_professionals(
    principals, crew, movie_ratings, names)
save_result(q8_people, "q8_multitalented_people", ["nconst"])
save_result(q8_projects.filter(F.col("role_count") >= 2), "q8_multiple_role_movies", ["nconst", "tconst"])
save_result(q8_comparison, "q8_per_person_comparison", ["nconst", "role_mode"])
save_result(q8_paired, "q8_paired_differences", [F.desc("multiple_minus_single"), "nconst"])
save_result(q8_overall, "q8_overall_comparison", ["comparable_people"])

q8_multitalented_people: 242,315 rows (complete CSV export)
+---------+-------------------------+----------------+
|nconst   |career_roles             |primaryName     |
+---------+-------------------------+----------------+
|nm0000005|[actor, director, writer]|Ingmar Bergman  |
|nm0000008|[actor, director, writer]|Marlon Brando   |
|nm0000009|[actor, director]        |Richard Burton  |
|nm0000010|[actor, director]        |James Cagney    |
|nm0000018|[actor, director, writer]|Kirk Douglas    |
|nm0000019|[actor, director, writer]|Federico Fellini|
|nm0000024|[actor, director]        |John Gielgud    |
|nm0000027|[actor, writer]          |Alec Guinness   |
|nm0000032|[actor, director, writer]|Charlton Heston |
|nm0000033|[actor, director, writer]|Alfred Hitchcock|
|nm0000036|[actor, director, writer]|Buster Keaton   |
|nm0000037|[actor, director, writer]|Gene Kelly      |
|nm0000040|[director, writer]       |Stanley Kubrick |
|nm0000041|[director, writer]       |Akira Kurosawa  |
|nm00

DataFrame[comparable_people: bigint, mean_person_level_rating_difference: double, median_person_level_rating_difference: double]